# Week 24, Day 3: Drift diffusion: accumulating evidence to a bound

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Wald's test in action

Simulate 5,000 SPRT trials with the truth H+ (samples of mean 0.2, sd 1) and bound A = ln(19). Accumulate the log likelihood ratio sample by sample for all trials at once until each crosses ±A, and store the error rate in `err_rate` and the mean number of samples in `mean_n`. A fixed-sample test that averages n samples errs with probability Φ(−0.2√n); store in `n_fixed` the smallest n giving the same error rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, log
m, s_, A = 0.2, 1.0, log(19)
r_s = np.random.default_rng(1947)
n_tr = 5000
L = np.zeros(n_tr); n_used = np.zeros(n_tr, int); done = np.zeros(n_tr, bool); choice = np.zeros(n_tr, int)
while not done.all():
    act = ~done
    x = r_s.normal(m, s_, act.sum())
    L[act] += ...                                 # log likelihood ratio of each sample
    n_used[act] += 1
    hit_up, hit_dn = act & (L >= A), act & (L <= -A)
    choice[hit_up], choice[hit_dn] = 1, -1
    done |= hit_up | hit_dn
err_rate = np.mean(choice == -1)
mean_n = n_used.mean()
Phi = lambda z: 0.5 * (1 + erf(z / sqrt(2)))
n_fixed = ...                                     # smallest n with Phi(-0.2 sqrt(n)) at most err_rate
print(f"error rate {err_rate:.3f};  SPRT uses {mean_n:.1f} samples on average, a fixed test needs {n_fixed}")

In [ ]:
import numpy as np
def test_error():
    "the error rate is between 0.02 and 0.06 for a target of 0.05"
    assert 0.02 < err_rate < 0.06

def test_faster():
    "the SPRT needs fewer samples on average than the fixed-sample test that achieves the same error rate"
    assert mean_n < n_fixed

test_error()
test_faster()
print("All checks passed")

### Exercise: Formulas against simulation

Write the formulas `acc_th(mu)` and `dt_th(mu)` for a = 1 and sigma = 1. Then, for mu = 0.5, 1 and 2, simulate 4,000 trials at once (dt = 0.001) until all hit a bound, and store the accuracy in `acc_sim[mu]` and the mean decision time in `dt_sim[mu]`.

In [ ]:
a_b, sig = 1.0, 1.0
def acc_th(mu): return ...
def dt_th(mu): return ...
mus = (0.5, 1.0, 2.0)
acc_sim, dt_sim = {}, {}
r_d = np.random.default_rng(1978)
for mu in mus:
    x = np.zeros(4000); t_hit = np.zeros(4000); won = np.zeros(4000, bool); live = np.ones(4000, bool); t = 0.0
    while live.any():
        t += 0.001
        x[live] += mu * 0.001 + sig * np.sqrt(0.001) * r_d.normal(size=live.sum())
        up, dn = live & (x >= a_b), live & (x <= -a_b)
        won[up] = True; t_hit[up | dn] = t; live &= ~(up | dn)
    acc_sim[mu], dt_sim[mu] = won.mean(), t_hit.mean()
for mu in mus:
    print(f"mu = {mu}: accuracy {acc_sim[mu]:.3f} (theory {acc_th(mu):.3f}), mean time {dt_sim[mu]:.3f} (theory {dt_th(mu):.3f})")

In [ ]:
import numpy as np
def test_acc():
    "simulated accuracy is within 0.02 of the formula for every drift"
    assert all(abs(acc_sim[mu] - acc_th(mu)) < 0.02 for mu in mus)

def test_dt():
    "simulated mean decision time is within 5% of the formula for every drift"
    assert all(abs(dt_sim[mu] - dt_th(mu)) / dt_th(mu) < 0.05 for mu in mus)

test_acc()
test_dt()
print("All checks passed")

### Exercise: The most rewarding caution

With mu = 1, sigma = 1 and T0 = 1 s, write `RR(a, T0=1.0)` from the formulas, evaluate it on a grid of a from 0.01 to 3 in steps of 0.001, and store the best a in `a_opt`. Repeat with T0 = 3 to get `a_opt_slow`.

In [ ]:
def RR(a, T0=1.0, mu=1.0):
    acc = 1 / (1 + np.exp(-2 * mu * a))
    dt_ = a / mu * np.tanh(mu * a)
    return ...
grid = np.arange(0.01, 3.0, 0.001)
a_opt = ...
a_opt_slow = grid[np.argmax([RR(a, 3.0) for a in grid])]
print(f"best bound {a_opt:.3f} (accuracy {1 / (1 + np.exp(-2 * a_opt)):.3f}) with T0 = 1 s;  {a_opt_slow:.3f} with T0 = 3 s")
plt.plot(grid, [RR(a) for a in grid], label="T0 = 1"); plt.plot(grid, [RR(a, 3.0) for a in grid], label="T0 = 3")
plt.xlabel("bound a"); plt.ylabel("reward rate"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_opt():
    "the optimal bound lies strictly inside the range, between 0.3 and 1.5"
    assert 0.3 < a_opt < 1.5

def test_peak():
    "the reward rate at the optimum beats both a very low (0.05) and a very high (3) bound"
    assert RR(a_opt) > RR(0.05) and RR(a_opt) > RR(3.0)

def test_T0():
    "a longer delay between trials (T0 = 3 s) makes the optimal bound higher"
    assert a_opt_slow > a_opt

test_opt()
test_peak()
test_T0()
print("All checks passed")